# Same climate calculation, three independent routes
### Python · CDO-only · NCO-only — Yiwen Li / Fresh Eyes on CMIP

**20-minute prepared demonstration · updated 9 November 2026**

Updated reference: [Evgenia Galytska's ESMValTool recipe](https://github.com/egalytska/Fresh_eyes/blob/040b9c7962355b1f8fa1312b7af867b458102b85/ESMValTool/recipe_temperature_anomalies.yml).
The [original Python workshop by Abhnil](https://github.com/abhnil/Fresh_Eyes_on_CMIP_Data_Workshop)
provides the broader climate-data processing context.

**Default: real ACCESS-CM2 historical + SSP5-8.5 output.** Both source files are
cached during the Binder build and are already staged on the prepared Mac.
The data mode is printed below and on the final figure. Missing real files
stop execution; the notebook never substitutes synthetic data.

For an offline software test, explicitly choose `WORKSHOP_MODE=synthetic`.

Run from top to bottom with **Shift+Enter**. Keep each `%%bash` header.
Python validates and draws the final comparison; CDO and NCO each perform their
own complete data processing from the same two raw inputs.

## 01 · Choose the data and create a fresh run (0:00–2:00)

The default `DATA_MODE` is `real`. Set `WORKSHOP_MODE=synthetic` only for an
explicit software test. The launch instructions also show how to supply local
file paths. The processing scripts select the needed dates before loading
arrays; the default run uses 552 months, not the full 1850–2100 archive.
Keep `END_YEAR = 2024` for the shared seminar. **2100 is an optional scenario
extension**, not the default comparison to Evgenia's 1979–2024 figures.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, uuid
from datetime import datetime, timezone
from IPython.display import display, Image

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts/run_updated_workflows.sh").exists():
    ROOT = ROOT.parent
assert (ROOT / "scripts/run_updated_workflows.sh").exists(), "Open this notebook from the workshop repository."
os.chdir(ROOT)

# The seminar uses real ACCESS-CM2. Synthetic mode is an explicit software-test option.
DATA_MODE = os.environ.get("WORKSHOP_MODE", "real")  # "real" or "synthetic"
END_YEAR = int(os.environ.get("WORKSHOP_END_YEAR", "2024"))
assert DATA_MODE in {"synthetic", "real"}
assert END_YEAR >= 2024
os.environ["OMP_NUM_THREADS"] = "1"
if DATA_MODE == "synthetic":
    fixture = ROOT / "data/updated_fixture"
    subprocess.run([sys.executable, "scripts/generate_updated_fixture.py", str(fixture)], check=True)
    HIST, FUTURE = fixture / "historical.nc", fixture / "ssp585.nc"
else:
    directory = ROOT / "data/access_cm2"
    HIST = Path(os.environ.get("WORKSHOP_HISTORICAL", str(directory / "tas_Amon_ACCESS-CM2_historical_r1i1p1f1_gn_185001-201412.nc")))
    FUTURE = Path(os.environ.get("WORKSHOP_FUTURE", str(directory / "tas_Amon_ACCESS-CM2_ssp585_r1i1p1f1_gn_201501-210012.nc")))
    missing = [str(p) for p in (HIST, FUTURE) if not p.is_file()]
    if missing:
        raise FileNotFoundError("Real-data mode requires both raw files. No synthetic substitution is allowed. Missing: " + "; ".join(missing) + ". See docs/UPDATED_DEMO.md.")

RUN = ROOT / "outputs" / ("updated_" + DATA_MODE + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6])
RUN.mkdir(parents=True)
os.environ.update(DEMO_HIST=os.path.relpath(HIST, ROOT), DEMO_FUTURE=os.path.relpath(FUTURE, ROOT), DEMO_OUT=os.path.relpath(RUN, ROOT), DEMO_END=str(END_YEAR))
print("DATA MODE:", "SYNTHETIC SOFTWARE TEST — NOT ACCESS-CM2" if DATA_MODE == "synthetic" else "REAL ACCESS-CM2: historical + SSP5-8.5")
print("Analysis:", f"1979–{END_YEAR}", "| Monthly reference: 1981–2010")
print("Outputs:", RUN.relative_to(ROOT))

## 02 · The calculation we are comparing

| Step | Definition used by every route |
|---|---|
| Input | ACCESS-CM2 monthly `tas` in Kelvin, historical + SSP5-8.5; or explicitly labeled synthetic test data |
| Main period | January 1979–December 2024: **552 months, 46 annual values** |
| Reference | January 1981–December 2010: 30 values for each calendar month |
| Monthly anomaly | Temperature minus the matching calendar-month reference field |
| Mean anomaly map | **Month-duration weighted** mean over the analysis period |
| Arctic mean | Latitude centres 65–90°N, all longitudes, **equal grid-point weights** |
| Annual stripes | **Equal weight for the 12 monthly samples** in each year |
| Comparison | Maximum absolute difference ≤ 0.0003°C, relative tolerance 0; coordinates and calendar labels agree |

The map and annual series deliberately use different time weights. This follows
`climate_statistics(period: full)` and `annual_statistics` in the updated
ESMValTool recipe. Replacing both with the same averaging operator changes the task.
The shared input check verifies matching grid, calendar and model/member metadata,
complete months, full calendar-month time bounds and finite input values.

In [ ]:
subprocess.run([sys.executable, "scripts/check_updated_inputs.py", str(HIST), str(FUTURE), "--end-year", str(END_YEAR)], check=True)
for command in (["cdo", "-V"], ["ncks", "--version"]):
    result = subprocess.run(command, check=True, text=True, capture_output=True)
    print((result.stdout + result.stderr).splitlines()[0])
print("Python", sys.version.split()[0])

## 03 · Python reference (2:00–5:00)

xarray concatenates the historical and future records, selects 1979–2024,
converts Kelvin to Celsius and groups the 1981–2010 reference by calendar month.
It subtracts that reference from each matching month. The mean map uses
`time_bnds` duration weights; the Arctic and annual reductions are arithmetic.

This route reads the raw inputs directly. It does not prepare input for either
command-line route. Open `scripts/run_python_updated.py` to inspect the full code.

In [ ]:
subprocess.run([sys.executable, "scripts/run_python_updated.py", str(HIST), str(FUTURE), str(RUN / "python"), "--end-year", str(END_YEAR)], check=True)

## 04 · CDO-only route (5:00–9:00)

| Command | Meaning |
|---|---|
| `selyear` → `mergetime` | Select the two experiment segments and join their months |
| `subc,273.15` | Convert Kelvin to Celsius |
| `selyear,1981/2010` → `ymonmean` | Create the 12 monthly reference fields |
| `ymonsub` | Subtract the field for the corresponding calendar month |
| `timsum -muldpm` divided by summed month lengths | Compute the duration-weighted mean map |
| `sellonlatbox,0,360,65,90` → `fldmean,weights=false` | Select the Arctic and average grid points equally |
| `yearmean` | Average the 12 monthly Arctic anomalies equally |

Read chained CDO operators from right to left. `-O` replaces generated outputs;
`-b F64` requests double precision. Each run has its own output folder.

In [ ]:
%%bash
set -euo pipefail
bash scripts/run_cdo_updated.sh "$DEMO_HIST" "$DEMO_FUTURE" "$DEMO_OUT/cdo" "$DEMO_END"

### Inspect one CDO step

The climatology has 12 fields, one per month; it is not a single reference mean.
The command below inspects the CDO result and changes no data.

In [ ]:
%%bash
set -euo pipefail
cdo -s ntime "$DEMO_OUT/cdo/climatology.nc"
cdo -s showyear "$DEMO_OUT/cdo/arctic_annual.nc"

## 05 · NCO-only route (9:00–13:30)

| Command | Meaning |
|---|---|
| `ncks` → `ncrcat` | Select historical/future records and concatenate them |
| `ncap2` | Convert units and subtract the matching monthly reference |
| `ncwa -d time,m,359,12` | Average one reference month across 30 years |
| `ncecat` | Assemble the 12 separate reference fields |
| `ncwa` with `time_bnds` duration weights | Create the weighted mean map |
| `ncwa -a lat,lon -d lat,65.0,90.0` | Compute an unweighted Arctic monthly series |
| `ncra --mro -d time,,,12,12` | Average consecutive groups of 12 months |

NCO time indices are zero based. January uses 0, 12, …, 348; February starts at
1. This bookkeeping requires January starts and complete consecutive years.
Bash controls loops and filenames; the numerical operations remain in NCO.
No CDO output is read by this route.

In [ ]:
%%bash
set -euo pipefail
bash scripts/run_nco_updated.sh "$DEMO_HIST" "$DEMO_FUTURE" "$DEMO_OUT/nco" "$DEMO_END"

## 06 · Numerical comparison (13:30–16:00)

Compare **five products** from each command-line route against Python:
monthly climatology, monthly anomaly fields, mean anomaly map, Arctic monthly
series and annual series. This gives ten product comparisons. The checks also
verify the baseline residual, time coverage, coordinates and equal-month annual
means. Similar-looking figures alone would not establish numerical agreement.

In [ ]:
subprocess.run([sys.executable, "scripts/compare_updated_workflows.py", str(RUN)], check=True)
report = json.loads((RUN / "updated_validation.json").read_text())
assert report["passed"]
print("Maximum absolute difference (°C):", max(row["max_absolute_difference_degC"] for row in report["comparisons"].values()))
print("Ten product comparisons passed; inspect the report for coordinate, baseline and annual checks.")

## 07 · Interpret the outputs (16:00–18:30)

Read each row across Python, CDO and NCO. Color limits are shared within the map
row and within the stripe row. The first row is a period-mean anomaly, not a
trend. Real mode is one model realization under historical forcing followed by
SSP5-8.5, not observations. Synthetic mode tests the software only.

These checks establish agreement among the three implementations. A comparison
with Evgenia's saved ESMValTool outputs is a separate verification step.

In [ ]:
display(Image(filename=str(RUN / "updated_comparison.png")))

## 08 · Hands-on practice and saving results (18:30–20:00)

Open [04_monthly_anomaly_hands_on.ipynb](04_monthly_anomaly_hands_on.ipynb).
Change one scientific choice at a time: Arctic boundary, monthly time weights,
or the scenario end year. State which calculation changed before comparing plots.

Your run folder contains each route's NetCDF outputs, `updated_validation.json`
and `updated_comparison.png`. In Binder, download results and your edited notebook
before leaving; the session is temporary. The older 1979–2014 notebooks are
retained with the `legacy_` prefix for provenance.